# 01 — Análisis Exploratorio de Datos (EDA)
**Equipo:** HEREDIA SARAVIA EDWIN · ILLANES CABALLERO JOEL ALEXANDER  
**Repositorio:** REPOSITORIO-TI26-TEAM-Heredia-Illanes  
**Dataset:** CRM Sales Opportunities (Kaggle)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from paths import CSV_FILES, RAW_DATA_DIR

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("Ruta de datos:", RAW_DATA_DIR)
assert RAW_DATA_DIR.exists(), f"No se encontró la carpeta: {RAW_DATA_DIR}"

In [ ]:
accounts = pd.read_csv(CSV_FILES["accounts"])
products = pd.read_csv(CSV_FILES["products"])
sales_teams = pd.read_csv(CSV_FILES["sales_teams"])
pipeline = pd.read_csv(CSV_FILES["sales_pipeline"])

print("Pipeline:", pipeline.shape)
print("Accounts:", accounts.shape)
print("Products:", products.shape)
print("Sales teams:", sales_teams.shape)
pipeline.head()

In [ ]:
print("=== Nulos en sales_pipeline ===")
print(pipeline.isnull().sum())
print("\n=== Etapas del pipeline ===")
print(pipeline["deal_stage"].value_counts())

In [ ]:
pipeline["deal_stage"].value_counts().plot(kind="bar", color="steelblue")
plt.title("Distribución de etapas del pipeline comercial")
plt.xlabel("deal_stage"); plt.ylabel("Cantidad")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
df = pipeline.merge(accounts, on="account", how="left")
df = df.merge(products, on="product", how="left")
df = df.merge(sales_teams, on="sales_agent", how="left")

print(f"Dataset enriquecido: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()

In [ ]:
closed = df[df["deal_stage"].isin(["Won", "Lost"])].copy()
closed["won"] = (closed["deal_stage"] == "Won").astype(int)

print("Distribución Won/Lost:")
print(closed["won"].value_counts(normalize=True))
closed.describe(include="all").T

In [ ]:
num_cols = [c for c in ["close_value", "revenue", "employees", "sales_price", "year_established"] if c in closed.columns]
corr = closed[num_cols + ["won"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlaciones — variables numéricas")
plt.tight_layout()
plt.show()

In [ ]:
if "sector" in closed.columns:
    pd.crosstab(closed["sector"], closed["deal_stage"], normalize="index").plot(
        kind="bar", stacked=True, figsize=(12, 5)
    )
    plt.title("Tasa Won/Lost por sector industrial")
    plt.ylabel("Proporción"); plt.xticks(rotation=45)
    plt.legend(title="Resultado")
    plt.tight_layout()
    plt.show()